# Fase 2: tiempos e imagen — memoria global


In [ ]:
%pip install nvcc4jupyter Pillow pandas
%load_ext nvcc4jupyter
!nvidia-smi
!nvcc --version


## Proyecto


In [ ]:
from pathlib import Path
import subprocess, os
root = Path('/content/P3-CUDA-Paralela')
if not root.exists():
    subprocess.run(['git', 'clone', '--branch', 'main',
                    'https://github.com/Jorge162017/P3-CUDA-Paralela.git', str(root)], check=True)
os.chdir(root)
subprocess.run(['git', 'checkout', 'main'], check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run(['make', 'build/hough'], check=True)
Path('results/fase2-global').mkdir(parents=True, exist_ok=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True))


## nvcc4jupyter


In [ ]:
%%cuda
#include <cuda_runtime.h>
#include <cstdio>
int main() {
    cudaDeviceProp gpu{};
    cudaError_t status = cudaGetDeviceProperties(&gpu, 0);
    if (status != cudaSuccess) {
        std::fprintf(stderr, "%s\n", cudaGetErrorString(status));
        return 1;
    }
    std::printf("GPU: %s\n", gpu.name);
    return 0;
}


## Diez mediciones con CUDA Events


In [ ]:
import pandas as pd
out = Path('results/fase2-global')
log = subprocess.check_output(['nvidia-smi'], text=True)
log += subprocess.check_output(['nvcc', '--version'], text=True)
log += 'Commit: ' + subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True)
log += 'Compilacion: make build/hough; -O2 -std=c++17 --fmad=false\n'
run = subprocess.run(['./build/hough', 'data/runway.pgm', '--runs', '10',
                      '--output', str(out)], text=True, capture_output=True)
log += run.stdout + run.stderr
(out / 'global-validacion.txt').write_text(log)
print(run.stdout)
if run.stderr:
    print(run.stderr)
assert run.returncode == 0, 'Fallo la ejecucion CUDA'
times = pd.read_csv(out / 'global-times.csv')
assert len(times) == 10 and times['validated'].all()
assert times['run'].tolist() == list(range(1, 11))
display(times[['run', 'kernel_ms', 'gpu', 'validated']])
summary = times['kernel_ms'].agg(['mean', 'std', 'min', 'max'])
summary.to_csv(out / 'global-summary.csv', header=['kernel_ms'])
display(summary.rename('kernel_ms').to_frame())


## Líneas detectadas


In [ ]:
from scripts.render_lines import render
from IPython.display import display, Image
THRESHOLD = None
MAX_LINES = 12
details = render('data/runway.pgm', out / 'global-accumulator.json',
                 out / 'global-lines.png', threshold=THRESHOLD, max_lines=MAX_LINES)
assert details['gpu_executed']
display(Image(filename=str(out / 'global-lines.png')))


## Resultados


In [ ]:
from google.colab import files
import zipfile
bundle = Path('build/fase2-global.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.glob('*')):
        if path.is_file():
            archive.write(path, path.as_posix())
files.download(str(bundle))
